In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('retail_ml.csv')
df

,Description_Encoded,Quantity,Revenue,Country_Encoded,Month,Year
0,24,12,83.40,20,12,2009
1,2869,12,81.00,20,12,2009
2,4436,12,81.00,20,12,2009
3,3174,48,100.80,20,12,2009
4,4059,24,30.00,20,12,2009
...,...,...,...,...,...,...
1036872,2669,12,10.20,8,12,2011
1036873,850,6,12.60,8,12,2011
1036874,856,4,16.60,8,12,2011
1036875,855,4,16.60,8,12,2011


In [3]:
df = df[df['Quantity'] <= 10000]
print(f"Rows after removing extremes: {len(df):,}")
print(f"Max Quantity now: {df['Quantity'].max()}")

Rows after removing extremes: 1,036,870
Max Quantity now: 10000


In [4]:
# Group into monthly totals
monthly = df.groupby(['Year', 'Month']).agg(
    Total_Revenue   = ('Revenue', 'sum'),
    Total_Orders    = ('Description_Encoded', 'count'),
    Total_Quantity  = ('Quantity', 'sum'),
    Unique_Products = ('Description_Encoded', 'nunique'),
).reset_index()

monthly = monthly.sort_values(['Year', 'Month']).reset_index(drop=True)
print(f"Total months: {len(monthly)}")
print(monthly[['Year', 'Month', 'Total_Revenue']])

# Add lag features
monthly['lag_1']     = monthly['Total_Revenue'].shift(1)
monthly['lag_2']     = monthly['Total_Revenue'].shift(2)
monthly['lag_3']     = monthly['Total_Revenue'].shift(3)
monthly['rolling_3'] = monthly['Total_Revenue'].shift(1).rolling(3).mean()

# Drop NaN rows created by lag
monthly = monthly.dropna().reset_index(drop=True)

print(f"\nMonths available for modelling: {len(monthly)}")
print(monthly.head())

Total months: 25
    Year  Month  Total_Revenue
0   2009     12      800939.83
1   2010      1      613824.59
2   2010      2      537813.35
3   2010      3      759179.09
4   2010      4      648886.07
5   2010      5      645627.64
6   2010      6      699172.30
7   2010      7      634886.18
8   2010      8      676059.87
9   2010      9      871859.66
10  2010     10     1097939.37
11  2010     11     1435532.81
12  2010     12     1187076.07
13  2011      1      594731.34
14  2011      2      508869.75
15  2011      3      691132.59
16  2011      4      516194.91
17  2011      5      741153.76
18  2011      6      738751.87
19  2011      7      689289.86
20  2011      8      725493.76
21  2011      9     1030468.78
22  2011     10     1106666.53
23  2011     11     1457742.24
24  2011     12      447020.38

Months available for modelling: 22
   Year  Month  Total_Revenue  Total_Orders  Total_Quantity  Unique_Products  \
0  2010      3      759179.09         40119          475205  

In [5]:
print(f"Total months: {len(monthly)}")
print(f"\nColumns: {monthly.columns.tolist()}")
print(f"\nMonthly revenue table:")
print(monthly[['Year', 'Month', 'Total_Revenue', 'lag_1', 'lag_2', 'lag_3', 'rolling_3']])

Total months: 22

Columns: ['Year', 'Month', 'Total_Revenue', 'Total_Orders', 'Total_Quantity', 'Unique_Products', 'lag_1', 'lag_2', 'lag_3', 'rolling_3']

Monthly revenue table:
    Year  Month  Total_Revenue       lag_1       lag_2       lag_3  \
0   2010      3      759179.09   537813.35   613824.59   800939.83   
1   2010      4      648886.07   759179.09   537813.35   613824.59   
2   2010      5      645627.64   648886.07   759179.09   537813.35   
3   2010      6      699172.30   645627.64   648886.07   759179.09   
4   2010      7      634886.18   699172.30   645627.64   648886.07   
5   2010      8      676059.87   634886.18   699172.30   645627.64   
6   2010      9      871859.66   676059.87   634886.18   699172.30   
7   2010     10     1097939.37   871859.66   676059.87   634886.18   
8   2010     11     1435532.81  1097939.37   871859.66   676059.87   
9   2010     12     1187076.07  1435532.81  1097939.37   871859.66   
10  2011      1      594731.34  1187076.07  1435532

In [6]:
# Remove last row — December 2011 is incomplete (only 9 days)
monthly = monthly.iloc[:-1].reset_index(drop=True)
print(f"Months after removing incomplete month: {len(monthly)}")
print(f"Last month now: {monthly['Year'].iloc[-1]}-{monthly['Month'].iloc[-1]:02d}")

Months after removing incomplete month: 21
Last month now: 2011-11


In [7]:
# Separate features and target
X = monthly[['Year', 'Month', 'Total_Orders', 
             'Total_Quantity', 'Unique_Products',
             'lag_1', 'lag_2', 'lag_3', 'rolling_3']]

y = monthly['Total_Revenue']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns: {X.columns.tolist()}")
print(f"\nTarget (first 5):")
print(y.head())

Features shape: (21, 9)
Target shape: (21,)

Feature columns: ['Year', 'Month', 'Total_Orders', 'Total_Quantity', 'Unique_Products', 'lag_1', 'lag_2', 'lag_3', 'rolling_3']

Target (first 5):
0    759179.09
1    648886.07
2    645627.64
3    699172.30
4    634886.18
Name: Total_Revenue, dtype: float64
